# 02 — First Generation (NaturalVoice AI, Phase 0)

**Purpose (PRD §7 Phase 0, tasks 6-8):** load the pretrained checkpoint, generate ONE short test recording, play + download it, and record model version + settings.

**Cost rules (PRD §11):** run `01_environment_check.ipynb` first; this notebook should also be a short single pass — one inference call, then shut the runtime down. No heavy runs here.

In [ ]:
import torch, yaml
from pathlib import Path

with open('configs/baseline.yaml') as fh:
    cfg = yaml.safe_load(fh)

model_id = cfg['model']['id']
print('checkpoint to load:', model_id, '| revision:', cfg['model'].get('revision') or 'unpinned')
assert torch.cuda.is_available(), 'GPU required — run 01_environment_check.ipynb first.'
print('CUDA device:', torch.cuda.get_device_name(0))

In [ ]:
import time
from voxcpm import VoxCPM

t0 = time.time()
model = VoxCPM.from_pretrained(model_id)
load_s = time.time() - t0
print(f'checkpoint {model_id} loaded successfully in {load_s:.1f}s')
assert model is not None, 'model failed to load (exit criterion failed)'

In [ ]:
import numpy as np
import soundfile as sf
import datetime

text = ('The ocean covers more than two thirds of our planet, yet most of it '
        'has never been seen by human eyes.')
t0 = time.time()
audio = model.infer(text, prompt='')
gen_s = time.time() - t0

audio = np.concatenate([np.asarray(w).reshape(-1) for w in (audio if isinstance(audio, list) else [audio])])
# NOTE (PRD §5.2): this generation uses the model's default voice — the one tested
# configuration for the MVP. No reference/wav was supplied (voice cloning needs
# documented consent, PRD §8), so no voice controls are simulated here.
sr = cfg['output']['sample_rate']  # confirm this is the model's actual output rate
print(f'generated {gen_s:.2f}s of audio | ~{len(audio) / sr:.2f}s at {sr} Hz')

Path('outputs').mkdir(exist_ok=True)
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
out_wav = Path(f'outputs/first_generation-{stamp}.wav')
sf.write(str(out_wav), audio, sr)
print('saved to', out_wav)

In [ ]:
# Exit criterion: audio can be played and downloaded.
from IPython.display import Audio, display

display(Audio(str(out_wav)))
try:
    from google.colab import files  # type: ignore
    files.download(str(out_wav))    # triggers the browser download
    print('download triggered.')
except Exception as exc:
    print(f'(not in Colab or download blocked: {type(exc).__name__} — audio is saved at {out_wav})')
dur = len(audio) / sr
print(f'Real-time factor: {gen_s:.2f}s generated for {dur:.2f}s of audio (RTF {gen_s / dur if dur else 0:.2f})')

In [ ]:
# Exit criterion: model version + settings recorded (PRD §5.5).
import json
import importlib.metadata as md

record = {
    'timestamp': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'run': 'first_generation',
    'model_id': model_id,
    'checkpoint_revision': cfg['model'].get('revision') or 'unpinned',
    'voxcpm_version': md.version('voxcpm'),
    'settings': {
        'sample_rate': sr,
        'seed': cfg['inference'].get('seed'),
        'cfg_value': cfg['inference'].get('cfg_value'),
        'inference_timesteps': cfg['inference'].get('inference_timesteps'),
    },
    'input_text': text,
    'input_chars': len(text),
    'gen_seconds': round(gen_s, 3),
    'output_seconds': round(len(audio) / sr, 3),
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'audio_file': str(out_wav),
}
with open('outputs/run_log.jsonl', 'a', encoding='utf-8') as fh:
    fh.write(json.dumps(record, ensure_ascii=False) + '\n')
print(json.dumps(record, indent=2, ensure_ascii=False))

## Phase 0 exit criteria — final check (PRD §7)

| Exit criterion | Satisfied by |
|---|---|
| Model loads successfully | cell `load-checkpoint` above |
| Test audio is generated | cell `generate-test-audio` |
| Audio can be played / downloaded | cell `play-download-audio` |
| Model version + settings recorded | `.gen/env_check-*.json` (notebook 01) + `outputs/run_log.jsonl` (this notebook) |

**Next step (cost rules 4-5, PRD §11):** commit the `.gen/` + `outputs/run_log.jsonl` record, then shut this GPU runtime down. Phase 1 (Gradio app) does not need a GPU running while it is being written — start it only for a short test.